# CFM56-5B — Gázkar Szimulátor & ECAM Panel
**Interactive throttle control** with A320-style ECAM engine display.

- **Throttle 0%** → idle (~1000 K T4)
- **Throttle 100%** → TOGA — full thrust (~1700 K T4)

*N1/N2 are estimated from empirical correlations. EGT, FF, OPR, Thrust are directly from the thermodynamic simulation.*

In [1]:
import sys
sys.path.insert(0, '..')

import importlib
import visualization.station_diagram as sd
import visualization.ts_diagram as ts
import visualization.model_3d as m3d
importlib.reload(sd)
importlib.reload(ts)
importlib.reload(m3d)

from engine import run_design_point
from visualization.station_diagram import plot_station_diagram
from visualization.ts_diagram import plot_ts_diagram
from visualization.model_3d import plot_3d_model
from IPython.display import display
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
from IPython.display import HTML

FLIGHT_PHASES = {
    'Takeoff   (0 ft, Mach 0.25)':        {'altitude_ft': 0,     'mach': 0.25, 'key': 'takeoff'},
    'Climb     (15 000 ft, Mach 0.50)':   {'altitude_ft': 15000, 'mach': 0.50, 'key': 'climb'},
    'Cruise    (35 000 ft, Mach 0.78)':   {'altitude_ft': 35000, 'mach': 0.78, 'key': 'cruise'},
}

# ── Empirical N1/N2 correlations for CFM56-5B ─────────────────────────────
# Real CFM56-5B data (AMM / Aircraft Commerce issue 58):
#   N1 idle ~22%,  N1 TOGA ~100%
#   N2 idle ~70%,  N2 TOGA ~100%
def estimate_n1(throttle_pct):
    return 22.0 + 0.78 * throttle_pct   # 22% idle → 100% TOGA

def estimate_n2(throttle_pct):
    return 70.0 + 0.30 * throttle_pct   # 70% idle → 100% TOGA

def compute_epr(result):
    """EPR = P_nozzle_exit / P_inlet  (direct from simulation)."""
    try:
        p_in  = result.stations['S2_inlet_exit'].P   # kPa
        p_out = result.stations['S8_core_nozz'].P    # kPa
        return p_out / p_in
    except Exception:
        return None

def gauge_svg(value, vmin, vmax, label, color='#00ff00', unit='', size=110):
    """Simple semicircular SVG gauge."""
    import math
    pct   = max(0.0, min(1.0, (value - vmin) / (vmax - vmin)))
    angle = -135 + pct * 270          # -135° (left) → +135° (right)
    rad   = math.radians(angle)
    cx, cy, r = size/2, size/2, size*0.38
    # needle tip
    nx = cx + r * math.sin(rad)
    ny = cy - r * math.cos(rad)
    # red arc from 90% onward
    warn_start = math.radians(-135 + 0.9 * 270)
    warn_end   = math.radians(135)
    wx1 = cx + r * math.sin(warn_start);  wy1 = cy - r * math.cos(warn_start)
    wx2 = cx + r * math.sin(warn_end);    wy2 = cy - r * math.cos(warn_end)

    val_str = f"{value:.1f}" if isinstance(value, float) else str(value)

    return f"""
    <svg width="{size}" height="{size}" style="display:inline-block;">
      <!-- background arc -->
      <path d="M {cx + r*math.sin(math.radians(-135)):.1f} {cy - r*math.cos(math.radians(-135)):.1f}
               A {r} {r} 0 1 1 {cx + r*math.sin(math.radians(135)):.1f} {cy - r*math.cos(math.radians(135)):.1f}"
            fill="none" stroke="#333" stroke-width="6" stroke-linecap="round"/>
      <!-- warning arc (red, >90%) -->
      <path d="M {wx1:.1f} {wy1:.1f} A {r} {r} 0 0 1 {wx2:.1f} {wy2:.1f}"
            fill="none" stroke="#cc2200" stroke-width="6" stroke-linecap="round"/>
      <!-- needle -->
      <line x1="{cx}" y1="{cy}" x2="{nx:.1f}" y2="{ny:.1f}"
            stroke="{color}" stroke-width="2.5" stroke-linecap="round"/>
      <circle cx="{cx}" cy="{cy}" r="3" fill="{color}"/>
      <!-- digital readout -->
      <rect x="{cx-22}" y="{cy+4}" width="44" height="18" rx="2"
            fill="none" stroke="{color}" stroke-width="1"/>
      <text x="{cx}" y="{cy+16}" text-anchor="middle"
            font-family="Courier New" font-size="11" fill="{color}">{val_str}</text>
      <!-- label -->
      <text x="{cx}" y="{size-4}" text-anchor="middle"
            font-family="Courier New" font-size="9" fill="#888">{label} {unit}</text>
    </svg>"""

def ecam_html(result, n1, n2):
    """A320-style ECAM with analog gauges + digital readouts."""
    egt_st = result.stations.get('S5_lpt_exit', result.stations.get('lpt_exit'))
    egt_c  = round(egt_st.T - 273.15) if egt_st else 0
    ff_kgh = round(result.fuel_flow * 3600)
    thr    = result.thrust_kN
    opr    = result.opr
    sfc    = result.sfc
    epr_v  = compute_epr(result)
    epr_s  = f"{epr_v:.3f}" if epr_v else "---"
    epr_f  = epr_v if epr_v else 1.0

    # gauges
    g_epr = gauge_svg(round(epr_f, 3), 1.0, 1.6,  'EPR',  '#00ff00', '', 115)
    g_egt = gauge_svg(egt_c,           0,   1000,  'EGT',  '#ffaa00', '°C', 115)
    g_n1  = gauge_svg(round(n1, 1),    0,   110,   'N1',   '#00ff00', '%', 115)

    return f"""
    <div style="background:#000; font-family:'Courier New',monospace;
                border:2px solid #555; border-radius:8px; padding:14px 20px;
                display:inline-block; min-width:560px; margin:8px 0;">

      <!-- Header -->
      <div style="display:flex; justify-content:space-around; color:#00aaff;
                  font-size:12px; letter-spacing:2px; border-bottom:1px solid #333;
                  padding-bottom:6px; margin-bottom:8px;">
        <span>── ENGINE 1 ──</span>
        <span>── ENGINE 2 ──</span>
      </div>

      <!-- Analog gauges row -->
      <div style="display:flex; justify-content:center; align-items:flex-end; gap:0px;">
        <!-- ENG 1 gauges -->
        <div style="display:flex; flex-direction:column; align-items:center;">
          {g_epr}
          {g_egt}
          {g_n1}
        </div>

        <!-- Center labels -->
        <div style="display:flex; flex-direction:column; justify-content:space-around;
                    align-items:center; height:345px; padding:0 12px;
                    color:#aaa; font-size:11px; letter-spacing:1px;">
          <span>EPR</span>
          <span>EGT<br><span style="font-size:9px;">°C</span></span>
          <span>N1<br><span style="font-size:9px;">%</span></span>
        </div>

        <!-- ENG 2 gauges (mirror) -->
        <div style="display:flex; flex-direction:column; align-items:center;">
          {g_epr}
          {g_egt}
          {g_n1}
        </div>
      </div>

      <!-- N2 row (digital only, like real ECAM) -->
      <div style="display:flex; justify-content:space-between; align-items:center;
                  border-top:1px solid #333; margin-top:6px; padding-top:6px;">
        <span style="font-size:18px; color:#00cc00;">{n2:.1f}</span>
        <span style="color:#aaa; font-size:11px; letter-spacing:1px;">── N2 % ──</span>
        <span style="font-size:18px; color:#00cc00;">{n2:.1f}</span>
      </div>

      <!-- Digital secondary params -->
      <div style="border-top:1px solid #333; margin-top:8px; padding-top:8px;">
        {''.join(f"""
        <div style="display:flex; justify-content:space-between; margin:3px 0;">
          <span style="color:#aaa; font-size:10px; width:50px;">{lbl}</span>
          <span style="font-size:15px; color:{col};">{v1}</span>
          <span style="font-size:15px; color:{col};">{v2}</span>
          <span style="color:#666; font-size:10px; width:60px; text-align:right;">{unit}</span>
        </div>""" for lbl, v1, v2, col, unit in [
            ('FF',  ff_kgh, ff_kgh, '#00e000', 'KG/H'),
            ('THR', f'{thr:.1f}', f'{thr:.1f}', '#00e000', 'kN'),
            ('OPR', f'{opr:.2f}', f'{opr:.2f}', '#00e000', ''),
            ('SFC', f'{sfc:.5f}', f'{sfc:.5f}', '#00cc00', 'kg/kN·s'),
        ])}
      </div>

      <div style="text-align:center; color:#444; font-size:8px; margin-top:8px;">
        EPR/EGT/FF/THR/OPR/SFC FROM SIMULATION · N1/N2 ESTIMATED
      </div>
    </div>
    """

# ── Widgets ───────────────────────────────────────────────────────────────
throttle_slider = widgets.IntSlider(
    value=100, min=0, max=100, step=5,
    description='Throttle [%]:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='420px')
)
phase_dropdown = widgets.Dropdown(
    options=list(FLIGHT_PHASES.keys()),
    value=list(FLIGHT_PHASES.keys())[0],
    description='Flight phase:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='420px')
)

controls = widgets.HBox([phase_dropdown, throttle_slider])
output   = widgets.Output()

def update(change):
    pct = throttle_slider.value / 100.0
    T4  = 1000.0 + pct * 700.0
    fp  = FLIGHT_PHASES[phase_dropdown.value]
    n1  = estimate_n1(throttle_slider.value)
    n2  = estimate_n2(throttle_slider.value)

    result = run_design_point(
        flight_phase=fp['key'],
        altitude_ft=fp['altitude_ft'],
        mach=fp['mach'],
        T4_override=T4,
    )

    with output:
        output.clear_output(wait=True)
        display(HTML(ecam_html(result, n1, n2)))

        fig1 = plot_station_diagram(result)
        plt.tight_layout(); display(fig1); plt.close(fig1)

        fig2 = plot_ts_diagram([result])
        plt.tight_layout(); display(fig2); plt.close(fig2)

        display(plot_3d_model(result))

throttle_slider.observe(update, names='value')
phase_dropdown.observe(update, names='value')

display(controls, output)
update(None)